In [0]:
CATALOG = "databricks_rr"      # catalog name (Free Edition = workspace)
SCHEMA = "default"         # schema name
VOLUME = "capstone"        # volume where the csv files are uploaded

DATA_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/project2"     # input csv files are here
OUTPUT_PATH = f"{DATA_PATH}/output"                              # exported csv files go here

In [0]:
# create the volume once (it is safe to run again)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
print("Volume ready. Upload sales.csv, products.csv and stores.csv into the folder project2 of this volume.")

Volume ready. Upload sales.csv, products.csv and stores.csv into the folder project2 of this volume.


In [0]:
# all tables below will be created in this catalog and schema
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

from pyspark.sql import functions as F
import os

In [0]:
# read everything as text first, then change the data types while cleaning
sales_raw = spark.read.csv(f"{DATA_PATH}/sales.csv", header=True)
products_raw = spark.read.csv(f"{DATA_PATH}/products.csv", header=True)
stores_raw = spark.read.csv(f"{DATA_PATH}/stores.csv", header=True)

print("Sales rows:", sales_raw.count())
print("Product rows:", products_raw.count())
print("Store rows:", stores_raw.count())
display(sales_raw.limit(5))

Sales rows: 3003
Product rows: 10
Store rows: 5


sale_id,store_id,product_id,sale_date,quantity,discount_pct,returned
1,3.0,7,2025-01-03,9,0,0
2,4.0,3,2025-02-26,4,10,0
3,4.0,4,2025-02-09,10,0,0
4,1.0,9,2025-02-21,2,10,0
5,2.0,2,2025-03-31,1,10,0


In [0]:
# sale_id is unique, so remove duplicate sale ids
sales = sales_raw.dropDuplicates(["sale_id"])

# change data types (try_cast and try_to_timestamp give null for wrong values like "abc" instead of error)
sales = sales.withColumn("store_id", F.expr("try_cast(try_cast(store_id as double) as int)"))
sales = sales.withColumn("product_id", F.expr("try_cast(product_id as int)"))
sales = sales.withColumn("quantity", F.expr("try_cast(quantity as int)"))
sales = sales.withColumn("discount_pct", F.expr("try_cast(discount_pct as double)"))
sales = sales.withColumn("returned", F.expr("try_cast(returned as int)"))
sales = sales.withColumn("sale_date", F.expr("try_to_timestamp(sale_date, 'yyyy-MM-dd')").cast("date"))

# missing discount means no discount, missing returned means not returned
sales = sales.fillna({"discount_pct": 0, "returned": 0})

# remove rows with missing important values and wrong quantity
sales = sales.dropna(subset=["sale_date", "store_id", "product_id", "quantity"])
sales = sales.filter(F.col("quantity") > 0)

print("Sales rows after cleaning:", sales.count())

Sales rows after cleaning: 2995


In [0]:
# clean the product and store tables
products = products_raw.withColumn("product_id", F.expr("try_cast(product_id as int)"))
products = products.withColumn("price", F.expr("try_cast(price as double)"))
products = products.withColumn("cost", F.expr("try_cast(cost as double)"))
products = products.dropna(subset=["product_id", "price", "cost"]).dropDuplicates(["product_id"])

stores = stores_raw.withColumn("store_id", F.expr("try_cast(store_id as int)"))
stores = stores.dropna(subset=["store_id"]).dropDuplicates(["store_id"])

In [0]:
# inner join with products (we need price and cost), left join with stores
sales_clean = sales.join(products, on="product_id", how="inner")
sales_clean = sales_clean.join(stores, on="store_id", how="left")

sales_clean = sales_clean.withColumn(
    "revenue", F.round(F.col("quantity") * F.col("price") * (1 - F.col("discount_pct") / 100), 2))
sales_clean = sales_clean.withColumn(
    "total_cost", F.round(F.col("quantity") * F.col("cost"), 2))
sales_clean = sales_clean.withColumn(
    "profit", F.round(F.col("revenue") - F.col("total_cost"), 2))
sales_clean = sales_clean.withColumn("month", F.date_format("sale_date", "yyyy-MM"))

sales_clean = sales_clean.select(
    "sale_id", "sale_date", "month", "store_id", "store_name", "region",
    "product_id", "product_name", "category", "quantity", "price", "cost",
    "discount_pct", "returned", "revenue", "total_cost", "profit")

print("Rows after joining:", sales_clean.count())
display(sales_clean.limit(10))

Rows after joining: 2995


sale_id,sale_date,month,store_id,store_name,region,product_id,product_name,category,quantity,price,cost,discount_pct,returned,revenue,total_cost,profit
56,2025-03-04,2025-03,1,Anna Nagar,South,6,Backpack,Accessories,6,1299.0,700.0,0.0,0,7794.0,4200.0,3594.0
73,2025-02-24,2025-02,4,Andheri,West,8,Notebook Pack,Stationery,8,249.0,110.0,20.0,0,1593.6,880.0,713.6
110,2025-02-01,2025-02,3,Whitefield,South,5,Water Bottle,Home,4,349.0,150.0,0.0,0,1396.0,600.0,796.0
120,2025-03-17,2025-03,4,Andheri,West,7,Desk Lamp,Home,9,899.0,500.0,20.0,0,6472.8,4500.0,1972.8
146,2025-02-22,2025-02,5,Banjara Hills,South,8,Notebook Pack,Stationery,8,249.0,110.0,0.0,0,1992.0,880.0,1112.0
149,2025-03-30,2025-03,3,Whitefield,South,10,Yoga Mat,Fitness,8,1099.0,600.0,10.0,0,7912.8,4800.0,3112.8
155,2025-03-06,2025-03,3,Whitefield,South,8,Notebook Pack,Stationery,4,249.0,110.0,10.0,1,896.4,440.0,456.4
156,2025-02-06,2025-02,5,Banjara Hills,South,2,Bluetooth Speaker,Electronics,3,2499.0,1600.0,10.0,0,6747.3,4800.0,1947.3
167,2025-01-27,2025-01,3,Whitefield,South,4,Running Shoes,Footwear,8,3299.0,2100.0,15.0,0,22433.2,16800.0,5633.2
168,2025-03-24,2025-03,3,Whitefield,South,6,Backpack,Accessories,9,1299.0,700.0,10.0,0,10521.9,6300.0,4221.9


In [0]:
# metrics for each product
product_sales = sales_clean.groupBy("product_id", "product_name", "category").agg(
    F.sum("quantity").alias("total_units_sold"),
    F.round(F.sum("revenue"), 2).alias("total_revenue"),
    F.round(F.sum("profit"), 2).alias("total_profit"),
    F.count("*").alias("total_orders"),
    F.sum("returned").alias("returned_orders"))

product_sales = product_sales.withColumn(
    "profit_margin_pct", F.round(F.col("total_profit") / F.col("total_revenue") * 100, 2))
product_sales = product_sales.withColumn(
    "return_rate_pct", F.round(F.col("returned_orders") / F.col("total_orders") * 100, 2))

display(product_sales.orderBy(F.col("total_units_sold").desc()))

product_id,product_name,category,total_units_sold,total_revenue,total_profit,total_orders,returned_orders,profit_margin_pct,return_rate_pct
3,Cotton T-Shirt,Apparel,2114,985225.6,520145.6,337,24,52.79,7.12
4,Running Shoes,Footwear,1917,5893663.5,1867963.5,320,18,31.69,5.63
9,Headphones,Electronics,1902,3547725.25,1265325.25,318,19,35.67,5.97
1,Wireless Mouse,Electronics,1898,1421141.35,567041.35,311,19,39.9,6.11
2,Bluetooth Speaker,Electronics,1835,4305152.25,1369152.25,301,17,31.8,5.65
6,Backpack,Accessories,1805,2185307.7,921807.7,302,27,42.18,8.94
5,Water Bottle,Home,1733,566060.55,306110.55,289,14,54.08,4.84
8,Notebook Pack,Stationery,1673,389622.75,205592.75,262,20,52.77,7.63
7,Desk Lamp,Home,1653,1384235.25,557735.25,275,15,40.29,5.45
10,Yoga Mat,Fitness,1643,1683503.15,697703.15,280,18,41.44,6.43


In [0]:
# profit margin by category
category_margin = sales_clean.groupBy("category").agg(
    F.sum("quantity").alias("total_units_sold"),
    F.round(F.sum("revenue"), 2).alias("total_revenue"),
    F.round(F.sum("profit"), 2).alias("total_profit"))

category_margin = category_margin.withColumn(
    "profit_margin_pct", F.round(F.col("total_profit") / F.col("total_revenue") * 100, 2))
category_margin = category_margin.orderBy(F.col("total_revenue").desc())

display(category_margin)

category,total_units_sold,total_revenue,total_profit,profit_margin_pct
Electronics,5635,9274018.85,3201518.85,34.52
Footwear,1917,5893663.5,1867963.5,31.69
Accessories,1805,2185307.7,921807.7,42.18
Home,3386,1950295.8,863845.8,44.29
Fitness,1643,1683503.15,697703.15,41.44
Apparel,2114,985225.6,520145.6,52.79
Stationery,1673,389622.75,205592.75,52.77


In [0]:
# monthly revenue by store
store_monthly_revenue = sales_clean.groupBy("store_id", "store_name", "region", "month").agg(
    F.round(F.sum("revenue"), 2).alias("monthly_revenue"),
    F.round(F.sum("profit"), 2).alias("monthly_profit"))
store_monthly_revenue = store_monthly_revenue.orderBy("store_id", "month")

display(store_monthly_revenue)

store_id,store_name,region,month,monthly_revenue,monthly_profit
1,Anna Nagar,South,2025-01,1930508.6,712328.6
1,Anna Nagar,South,2025-02,1650611.5,606781.5
1,Anna Nagar,South,2025-03,1787623.2,680043.2
2,T Nagar,South,2025-01,1938204.05,707894.05
2,T Nagar,South,2025-02,1834270.35,667000.35
2,T Nagar,South,2025-03,2185660.4,802020.4
3,Whitefield,South,2025-01,1432767.2,519247.2
3,Whitefield,South,2025-02,1216624.3,463294.3
3,Whitefield,South,2025-03,1542551.65,591481.65
4,Andheri,West,2025-01,1625451.4,585981.4


In [0]:
# save as Delta tables (managed tables in the catalog)
sales_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("sales_clean")
product_sales.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("product_sales")
category_margin.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("category_margin")
store_monthly_revenue.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("store_monthly_revenue")
print("Delta tables saved: sales_clean, product_sales, category_margin, store_monthly_revenue")

Delta tables saved: sales_clean, product_sales, category_margin, store_monthly_revenue


In [0]:
# export the final metrics as single csv files
os.makedirs(OUTPUT_PATH, exist_ok=True)
product_sales.toPandas().to_csv(f"{OUTPUT_PATH}/product_sales.csv", index=False)
category_margin.toPandas().to_csv(f"{OUTPUT_PATH}/category_margin.csv", index=False)
store_monthly_revenue.toPandas().to_csv(f"{OUTPUT_PATH}/store_monthly_revenue.csv", index=False)
print("CSV files saved in:", OUTPUT_PATH)

CSV files saved in: /Volumes/databricks_rr/default/capstone/project2/output


In [0]:
%sql
SELECT product_name, category, total_units_sold, total_revenue
FROM product_sales
ORDER BY total_units_sold DESC
LIMIT 4

product_name,category,total_units_sold,total_revenue
Cotton T-Shirt,Apparel,2114,985225.6
Running Shoes,Footwear,1917,5893663.5
Headphones,Electronics,1902,3547725.25
Wireless Mouse,Electronics,1898,1421141.35
